# Case Study 2: SaaS Customer Churn Prediction

Predict customer churn and derive business recommendations for a SaaS company.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
np.random.seed(42)
print("All imports successful.")

## Dataset Generation

In [ ]:
def generate_saas_churn_data(n_customers=1000, random_state=42):
    np.random.seed(random_state)
    tenure = np.random.randint(1, 73, n_customers)
    monthly_charges = np.random.uniform(15, 150, n_customers)
    total_charges = tenure * monthly_charges * np.random.uniform(0.95, 1.05, n_customers)
    contract = np.random.choice(['Month-to-month', 'One year', 'Two year'],
                                 n_customers, p=[0.5, 0.3, 0.2])
    payment = np.random.choice(['Credit card', 'Bank transfer', 'Invoice'],
                                n_customers, p=[0.4, 0.3, 0.3])
    logins = np.clip(np.random.exponential(10, n_customers), 0, 30)
    usage = np.clip(np.random.exponential(25, n_customers), 0, 80)
    tickets = np.random.poisson(2, n_customers)
    tickets = np.clip(tickets + np.random.binomial(3, 0.3, n_customers), 0, 15)
    active_features = np.random.randint(1, 21, n_customers)
    churn_prob = 0.05
    churn_prob += 0.25 * (tenure < 12).astype(float)
    churn_prob += 0.15 * (monthly_charges > 100).astype(float)
    churn_prob += 0.20 * (np.array(contract) == 'Month-to-month').astype(float)
    churn_prob += 0.15 * (tickets > 5).astype(float)
    churn_prob -= 0.10 * (active_features > 12).astype(float)
    churn_prob -= 0.10 * (usage > 40).astype(float)
    churn_prob = np.clip(churn_prob, 0.02, 0.80)
    churn = np.random.binomial(1, churn_prob)
    df = pd.DataFrame({
        'CustomerID': [f'C_{i+1:04d}' for i in range(n_customers)],
        'Tenure_Months': tenure,
        'Monthly_Charges': monthly_charges.round(2),
        'Total_Charges': total_charges.round(2),
        'Contract_Type': contract,
        'Payment_Method': payment,
        'Avg_Weekly_Logins': logins.round(1),
        'Avg_Weekly_Usage_Hours': usage.round(1),
        'Support_Tickets': tickets,
        'Active_Features': active_features,
        'Churn': churn
    })
    return df

df = generate_saas_churn_data()
print(f"Dataset created: {df.shape[0]} customers, {df.shape[1]} columns")

## Task 1: Data Loading and Initial Exploration

In [ ]:
print("=== First 5 rows ===")
display(df.head())

print("\n=== Data types ===")
print(df.dtypes)

print("\n=== Basic statistics ===")
display(df.describe())

print("\n=== Missing values ===")
print(df.isnull().sum().sum(), "missing values found")

print("\n=== Dataset shape ===")
print(f"{df.shape[0]} customers, {df.shape[1]} columns")

print("\n=== Churn rate ===")
churn_rate = df['Churn'].mean() * 100
print(f"{churn_rate:.1f}% of customers have churned")

## Task 2: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

churn_counts = df['Churn'].value_counts()
axes[0, 0].bar(['Active', 'Churned'], churn_counts.values,
               color=['steelblue', 'crimson'])
axes[0, 0].set_title(f'Churn Distribution ({churn_rate:.1f}% churn)')
axes[0, 0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0, 0].text(i, v + 5, str(v), ha='center')

num_cols = ['Tenure_Months', 'Monthly_Charges', 'Total_Charges',
            'Avg_Weekly_Logins', 'Avg_Weekly_Usage_Hours', 'Support_Tickets', 'Active_Features']
melted = df.melt(id_vars=['Churn'], value_vars=num_cols,
                 var_name='Feature', value_name='Value')
sns.boxplot(data=melted, x='Feature', y='Value', hue='Churn', ax=axes[0, 1])
axes[0, 1].set_title('Feature Distributions by Churn Status')
axes[0, 1].tick_params(axis='x', rotation=45)

corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=axes[0, 2])
axes[0, 2].set_title('Feature Correlation Matrix')

contract_churn = df.groupby('Contract_Type')['Churn'].mean() * 100
axes[1, 0].bar(contract_churn.index, contract_churn.values, color=['crimson', 'steelblue', 'green'])
axes[1, 0].set_title('Churn Rate by Contract Type')
axes[1, 0].set_ylabel('Churn Rate (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

payment_churn = df.groupby('Payment_Method')['Churn'].mean() * 100
axes[1, 1].bar(payment_churn.index, payment_churn.values, color=['steelblue', 'coral', 'green'])
axes[1, 1].set_title('Churn Rate by Payment Method')
axes[1, 1].set_ylabel('Churn Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for cls, color, label in [(0, 'steelblue', 'Active'), (1, 'crimson', 'Churned')]:
        subset = df[df['Churn'] == cls][col]
        sns.kdeplot(subset, fill=True, alpha=0.3, color=color,
                    label=label, ax=axes[i])
    axes[i].set_title(f'{col} by Churn Status')
    axes[i].legend()

for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

### EDA Insights

- Month-to-month contracts have the highest churn rate (~40%). Long-term contracts retain customers.
- Customers with short tenure (<12 months) are much more likely to churn.
- Higher monthly charges correlate with higher churn.
- More support tickets are associated with higher churn.
- Higher usage (logins, hours, active features) is associated with lower churn.
- Payment method has a weaker but noticeable effect on churn.

## Task 3: Data Preprocessing

In [ ]:
df_ml = df.drop(columns=['CustomerID'])

df_ml = pd.get_dummies(df_ml, columns=['Contract_Type', 'Payment_Method'],
                        drop_first=True)

feature_cols = [c for c in df_ml.columns if c != 'Churn']
X = df_ml[feature_cols].values
y = df_ml['Churn'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate: {y_test.mean()*100:.1f}%")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features standardized.")

## Task 4: Model Training

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_pred_lr_train = lr.predict(X_train_scaled)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
y_pred_rf_train = rf.predict(X_train_scaled)

gbt = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbt.fit(X_train_scaled, y_train)
y_pred_gbt = gbt.predict(X_test_scaled)
y_pred_gbt_train = gbt.predict(X_train_scaled)

print("=== Model Accuracy Comparison ===")
print(f"Logistic Regression - Train: {accuracy_score(y_train, y_pred_lr_train):.3f}")
print(f"Logistic Regression - Test:  {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Random Forest - Train:       {accuracy_score(y_train, y_pred_rf_train):.3f}")
print(f"Random Forest - Test:        {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Gradient Boosting - Train:   {accuracy_score(y_train, y_pred_gbt_train):.3f}")
print(f"Gradient Boosting - Test:    {accuracy_score(y_test, y_pred_gbt):.3f}")

## Task 5: Model Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f'Logistic Regression\nAcc: {accuracy_score(y_test, y_pred_lr):.3f}')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Random Forest\nAcc: {accuracy_score(y_test, y_pred_rf):.3f}')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

cm_gbt = confusion_matrix(y_test, y_pred_gbt)
sns.heatmap(cm_gbt, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title(f'Gradient Boosting\nAcc: {accuracy_score(y_test, y_pred_gbt):.3f}')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("\n=== Classification Report: Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['Active', 'Churned']))

print("\n=== Classification Report: Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=['Active', 'Churned']))

print("\n=== Classification Report: Gradient Boosting ===")
print(classification_report(y_test, y_pred_gbt, target_names=['Active', 'Churned']))

In [ ]:
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]
y_prob_gbt = gbt.predict_proba(X_test_scaled)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_gbt, tpr_gbt, _ = roc_curve(y_test, y_prob_gbt)

auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)
auc_gbt = roc_auc_score(y_test, y_prob_gbt)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})', lw=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', lw=2)
plt.plot(fpr_gbt, tpr_gbt, label=f'Gradient Boosting (AUC = {auc_gbt:.3f})', lw=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.xlim([0, 1])
plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
metrics = {
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gbt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gbt)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gbt)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gbt)
    ],
    'ROC AUC': [auc_lr, auc_rf, auc_gbt],
}
metrics_df = pd.DataFrame(
    metrics, index=['Logistic Regression', 'Random Forest', 'Gradient Boosting']
)

ax = metrics_df.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_ylim([0.5, 1.0])
ax.legend(loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)
plt.tight_layout()
plt.show()

print("\n=== Metrics Summary ===")
display(metrics_df)

## Task 6: Feature Importance Analysis

In [ ]:
feature_names = feature_cols

rf_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

gbt_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': gbt.feature_importances_
}).sort_values('Importance', ascending=True)

lr_coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].barh(rf_importance_df['Feature'][-10:],
             rf_importance_df['Importance'][-10:], color='steelblue')
axes[0].set_title('Random Forest: Top 10 Features')
axes[0].set_xlabel('Importance')

axes[1].barh(gbt_importance_df['Feature'][-10:],
             gbt_importance_df['Importance'][-10:], color='green')
axes[1].set_title('Gradient Boosting: Top 10 Features')
axes[1].set_xlabel('Importance')

colors = ['crimson' if c < 0 else 'steelblue' for c in lr_coef_df['Coefficient']]
top10_lr = lr_coef_df.iloc[:5]._append(lr_coef_df.iloc[-5:])
colors_top = ['crimson' if c < 0 else 'steelblue' for c in top10_lr['Coefficient']]
axes[2].barh(top10_lr['Feature'], top10_lr['Coefficient'], color=colors_top)
axes[2].set_title('Logistic Regression: Top Coefficients')
axes[2].set_xlabel('Coefficient')
axes[2].axvline(0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n=== Top 3 Drivers of Churn ===")
print("1. Contract type (Month-to-month) — customers on monthly contracts churn most")
print("2. Tenure — newer customers (under 12 months) are high risk")
print("3. Support tickets — high ticket volume signals frustration")

## Task 7: Business Recommendations & Cost-Benefit Analysis

In [ ]:
print("=" * 60)
print("COST-BENEFIT ANALYSIS: CUSTOMER RETENTION PROGRAM")
print("=" * 60)

cac = 500
monthly_revenue = 75
avg_lifetime = 36
ltv = monthly_revenue * avg_lifetime
retention_discount = 90

churned_total = df['Churn'].sum()
churn_cost_total = churned_total * cac

identified_churn = int(churned_total * 0.60)
saved_customers = int(identified_churn * 0.40)

cost_of_program = identified_churn * retention_discount
savings_from_retained = saved_customers * ltv
savings_from_acquisition = saved_customers * cac
total_savings = savings_from_retained + savings_from_acquisition
net_benefit = total_savings - cost_of_program
roi = (net_benefit / cost_of_program) * 100

print(f"\nAssumptions:")
print(f"  Customer Acquisition Cost (CAC): ${cac}")
print(f"  Monthly Revenue per Customer: ${monthly_revenue}")
print(f"  Average Customer Lifetime: {avg_lifetime} months")
print(f"  Customer Lifetime Value (LTV): ${ltv}")
print(f"  Retention Discount Cost per Targeted Customer: ${retention_discount}")
print(f"\nChurn Statistics:")
print(f"  Total churned in dataset: {churned_total}")
print(f"  Total churn cost (lost acquisition): ${churn_cost_total:,.0f}")
print(f"\nRetention Program Projections:")
print(f"  Churn customers identified by model: {identified_churn} (60% of total)")
print(f"  Customers saved: {saved_customers} (40% of targeted)")
print(f"  Cost of retention program: ${cost_of_program:,.0f}")
print(f"  Savings from retained LTV: ${savings_from_retained:,.0f}")
print(f"  Savings from avoided acquisition: ${savings_from_acquisition:,.0f}")
print(f"\nFinal Analysis:")
print(f"  Total savings: ${total_savings:,.0f}")
print(f"  Program cost: ${cost_of_program:,.0f}")
print(f"  Net benefit: ${net_benefit:,.0f}")
print(f"  ROI: {roi:,.0f}%")

### Business Recommendations

**1. Convert month-to-month contracts to annual plans.**
Offer a 10–15% discount for customers who switch from month-to-month to annual contracts. This alone could reduce churn significantly based on the feature importance analysis.

**2. Improve onboarding for new customers (first 90 days).**
Tenure is a top predictor of churn. Implement a structured onboarding program with check-ins, training webinars, and usage milestones to drive early engagement.

**3. Proactive support for high-ticket customers.**
Monitor support ticket volume. When a customer exceeds 3 tickets in a month, trigger a proactive outreach from a customer success manager.

### Cost-Benefit Conclusion

The retention program delivers a strong positive ROI. Investing in targeted retention for at-risk customers is significantly cheaper than acquiring new customers.